In [ ]:
!pip install -r requirements.txt

In [24]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig, TrainingArguments, Trainer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import evaluate
import pandas as pd
import os


In [7]:
# 1. Load training data from CSV
data_path = "./data/sample_training_data.csv"
df = pd.read_csv(data_path)

print("Data shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Column dtypes:")
print(df.dtypes)
print("\nFirst few rows:")
print(df.head(10))
print("\nData info:")
print(df.info())


Data shape: (10, 2)
Columns: ['text', 'label']
Column dtypes:
text       str
label    int64
dtype: object

First few rows:
                    text  label
0  我想去北京旅游因为我想看看真正的"首都"。      1
1               这个笑话很有趣。      0
2   小王很喜欢喝茶，因为他认为茶能"醒脑"。      0
3    今天天气真好，让人想起了"晴空万里"。      0
4          他说这项工作很"费脑子"。      0
5         这本书讲的是"书虫"的故事。      1
6         我最喜欢在"图书馆"里读书。      0
7           她说这个计划很"周密"。      0
8     他笑着说："我真是'心想事成'啊！"      0
9           这是一个"火爆"的话题。      0

Data info:
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    10 non-null     str  
 1   label   10 non-null     int64
dtypes: int64(1), str(1)
memory usage: 705.0 bytes
None


In [8]:
# 2. Handle labels
df["label"] = df["label"].astype(int)

# Check class distribution
class_counts = df["label"].value_counts()
print("\nClass distribution:")
print(class_counts)
print("Class percentages:")
print(class_counts / len(df))




Class distribution:
label
0    8
1    2
Name: count, dtype: int64
Class percentages:
label
0    0.8
1    0.2
Name: count, dtype: float64


In [9]:
# 3. Train/Val split
# Ensure label column is properly formatted
df = df.dropna(subset=['text', 'label'])  # Remove any rows with NaN
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(int)

# Perform train/val split with stratification
X_train, X_val, y_train, y_val = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)
    
# Create dataframes
train_df = pd.DataFrame({"text": X_train.values, "label": y_train.values})
val_df = pd.DataFrame({"text": X_val.values, "label": y_val.values})
    
# Create HuggingFace dataset
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False)
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2
    })
})


In [11]:
# ============================================================
# DEVICE CONFIGURATION - Set USE_GPU to True if GPU available
# ============================================================
import torch

USE_GPU = False  # Set to True if you have GPU (CUDA/MPS)

# Detect available device
if USE_GPU:
    if torch.cuda.is_available():
        device = "cuda"
        print("✅ GPU (CUDA) detected and enabled")
    elif torch.backends.mps.is_available():
        device = "mps"  # Mac with Apple Silicon
        print("✅ GPU (MPS) detected and enabled")
    else:
        device = "cpu"
        print("⚠️ GPU requested but not available, falling back to CPU")
else:
    device = "cpu"
    print("✅ Using CPU (set USE_GPU=True to enable GPU)")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device.upper()}")

✅ Using CPU (set USE_GPU=True to enable GPU)
PyTorch version: 2.10.0
Device: CPU


## PIYING PREPROCESSING

In [13]:
from pypinyin import pinyin, Style
map = {'ㄅ': 'p', 'ㄆ': 'ph', 'ㄇ': 'm', 'ㄈ': 'f', 'ㄉ': 't','ㄊ': 'th', 'ㄋ': 'n', 
       'ㄌ': 'l', 'ㄍ': 'k', 'ㄎ': 'kh', 'ㄏ': 'h', 'ㄐ': 'ts', 'ㄑ': 'tsh', 'ㄒ': 's', 
       'ㄓ': 'tsr', 'ㄔ': 'tshr', 'ㄕ': 'sr', 'ㄖ': 'jr', 'ㄗ': 'ts', 'ㄘ': 'tsh', 'ㄙ': 's',
       'ㄚ': 'a', 'ㄛ': 'o', 'ㄜ': 'o', 'ㄝ': 'e', 'ㄞ': 'ai', 'ㄟ': 'ei', 'ㄠ': 'au', 'ㄡ': 'ou', 
       'ㄢ': 'an', 'ㄣ': 'en', 'ㄤ': 'ang', 'ㄥ': 'eng', 'ㄦ': 'er', 'ㄧ': 'i', 'ㄨ': 'u', 'ㄩ': 'yu'}

def handle_chinese(text):
    result = pinyin(text, style=Style.BOPOMOFO)
    print(result)
    
    piying = ""
    for char in result[0][0]:
        if(map.get(char) is not None):
            piying += map[char]
    print(piying)
    
    return piying

In [14]:
import pronouncing

alpabet_to_pinyin = {
  "B": "ㄅ", "P": "ㄆ", "M": "ㄇ", "F": "ㄈ",
  "D": "ㄉ", "T": "ㄊ", "N": "ㄋ", "L": "ㄌ",
  "G": "ㄍ", "K": "ㄎ", "HH": "ㄏ",
  "JH": "ㄐ", "CH": "ㄔ", "ZH": "ㄓ",
  "SH": "ㄕ", "S": "ㄙ", "Z": "ㄗ",
  "TH":  "ㄊ", "DH": "ㄉ",  
  "V": "ㄈ", "W": "ㄨ", "Y": "ㄧ", "R": "ㄖ", 
  "NG": "ㄥ", "AA": "ㄚ", "AE": "ㄝ",    
  "AH": "ㄜ", "AO": "ㄛ",
  "AW": "ㄠ","AY": "ㄞ", "EH": "ㄟ",
  "EY": "ㄟ", "IH": "ㄧ", "IY": "ㄧ",
  "OW": "ㄡ",  "OY": "ㄡㄧ",
  "ER": "ㄦ", "UH": "ㄨ", "UW": "ㄨ",
}


def handle_english(text):
    phones = pronouncing.phones_for_word(text)
    print(phones)

    for phone in phones:
        pinyin_representation = []
        for symbol in phone.split():
            base_symbol = symbol.rstrip("012")  # Remove stress markers
            if base_symbol in alpabet_to_pinyin:
                pinyin_representation.append(map[alpabet_to_pinyin[base_symbol]])
                
    print("".join(pinyin_representation))
    return "".join(pinyin_representation)


In [15]:
def piying_preprocessing(text):
    '''
    Preprocess the input text for tailo piying classification.
    Chinese/ Taiwanese-> 注音 -> tailo piying 
    English -> phonetic symbol spelling
    return piying
    '''
    #English
    if(text.isascii()):
        handle_english(text.lower())
        return handle_english
    else:
        handle_chinese(text)

In [16]:
piying_preprocessing("hello")

['HH AH0 L OW1', 'HH EH0 L OW1']
heilou


<function __main__.handle_english(text)>

## Context Model

### Define Model

In [ ]:
import torch.nn as nn
from transformers import BertModel, BertPreTrainedModel, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback, PreTrainedConfig


In [ ]:
class PinyinBertClassifier(nn.Module):
    def __init__(self, config, pinyin_vocab_size):
        super().__init__()
        # We store the config so the Trainer can still access it if needed
        self.config = config 
        
        # Initialize the base BERT model
        self.bert = BertModel.from_pretrained("bert-base-chinese", config=config)
        
        # Pinyin Embedding Layer
        self.pinyin_embeddings = nn.Embedding(pinyin_vocab_size, config.hidden_size)
        
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)

    def forward(self, input_ids, attention_mask=None, pinyin_ids=None, labels=None, **kwargs):
        # 1. Get Semantic features
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state # [batch, seq_len, 768]
        
        # 2. Get Phonetic features
        # If pinyin_ids is None during a random test, this would fail, 
        # but the Trainer will provide it.
        p_embeds = self.pinyin_embeddings(pinyin_ids)
        
        # 3. Fusion (Addition)
        fused_output = sequence_output + p_embeds
        
        # 4. Pooling (Taking the [CLS] token at index 0)
        pooled_output = fused_output[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)

        # 5. Trainer-compatible output
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.config.num_labels), labels.view(-1))

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}

### Data Preprocessing

In [22]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
# Example Pinyin Vocab (You should build this from your unique Pinyin set)
pinyin_vocab = {"[PAD]": 0, "[UNK]": 1} 

def preprocess_function(examples):
    # Standard Tokenization
    result = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)
    
    # Custom Pinyin Processing using your function
    all_pinyin_ids = []
    for text in examples["text"]:
        # Use your function
        p_string = piying_preprocessing(text) 
        tokens = p_string.split() # Adjust based on your function's return type
        
        # Map to IDs and pad/truncate to 128
        ids = [pinyin_vocab.get(p, 1) for p in tokens]
        ids = [0] + ids[:126] + [0] # Account for [CLS] and [SEP]
        ids += [0] * (128 - len(ids))
        all_pinyin_ids.append(ids)
        
    result["pinyin_ids"] = all_pinyin_ids
    return result

# dataset = Dataset.from_dict(your_data).map(preprocess_function, batched=True)

### Train

In [ ]:
# 6. Metrics
accuracy = evaluate.load("accuracy")
f1_macro = evaluate.load("f1")
    
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_macro.compute(predictions=preds, references=labels, average="macro")["f1"]
    }

In [32]:
output_dir = "./models/toxic_classifier"
os.makedirs(output_dir, exist_ok=True)
    
# Adjust batch sizes based on device
if device == "cpu":
    batch_size_train = 2
    batch_size_eval = 2
    num_epochs = 2
else:  # GPU
    batch_size_train = 16
    batch_size_eval = 32
    num_epochs = 5
    
args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size_train,
    per_device_eval_batch_size=batch_size_eval,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    seed=42,
    logging_steps=1
)
    


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [35]:
# Initialize Model
config = AutoConfig.from_pretrained("bert-base-chinese", num_labels=2)
#piying_sz = len(pinyin_vocab)
piying_sz = 5000
model = PinyinBertClassifier(config, pinyin_vocab_size=piying_sz)


# Your original Trainer setup
trainer = Trainer(
    model=model,
    args=args, # Use the args from your snippet
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1269.61it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1269.61it/s, Materializing param=pooler.dense.weight]                
BertModel

TypeError: PinyinBertClassifier.forward() missing 1 required positional argument: 'input_ids'

### Inference

In [36]:
def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
    
    # Process Pinyin for a single sentence
    p_string = piying_preprocessing(text)
    p_ids = torch.tensor([[pinyin_vocab.get(p, 1) for p in p_string.split()]])
    # ... (Add padding/CLS/SEP logic same as training) ...

    with torch.no_grad():
        out = model(input_ids=inputs["input_ids"], pinyin_ids=p_ids)
        return torch.argmax(out["logits"], dim=-1)

The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.


In [ ]:
predict("你的笑话真有趣")

In [ ]:
# 11. Confusion Matrix & Classification Report
predictions = trainer.predict(dataset["validation"])
y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=[0, 1], target_names=["Non-Pun", "Pun"]))

# Plot confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Non-Pun", "Pun"],
            yticklabels=["Non-Pun", "Pun"],
            cbar_kws={'label': 'Count'})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Pun Classification")
plt.tight_layout()
plt.show()

print("\n✅ Model training completed!")
print(f"Model saved to: {output_dir}")